# Feature Selection

## Imports

In [ ]:
import pandas as pd
import numpy as np
import math

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, SelectKBest, SelectFromModel, chi2, f_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.svm import LinearSVR
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt

## Load the dataset

In [ ]:
# Load the dataset
df = pd.read_csv('俄罗斯_汇总.csv')

# Print datatypes
print(df.dtypes)

# Describe columns
df.describe(include='all')

In [ ]:
# Preview the dataset
df.head()

## Remove Unwanted Features

In [ ]:
# Check if there are null values in any of the columns. Find the ones with lots null values and remove them next
df.isna().sum()

In [ ]:
# Remove not needed columns
columns_to_remove = ['日期', 'GPR', 'GPRD_MA30', 'Balance of Payments:Primary:Current Account:Goods and Services','Actual Market Price:Global:Natural Gas:Russian Exports to Germany','Japan Spot Price:LNG (Indonesian production):Japan','German Spot Price:Natural Gas (Russian):German Ports','Actual market price:Global:Wheat']
df.drop(columns_to_remove, axis=1, inplace=True)

# Check that the columns are indeed dropped
df.head()

## Model Performance

Next, split the dataset into feature vectors `X` and target vector (基准利率) `Y` to fit a RandomForestRegressor model. Then compare the performance of each feature selection technique, using [R-square], [RMSE], and [MAE] as evaluation metrics.

In [ ]:
# Split feature and target vectors
X = df.drop("GPRC_RUS", axis=1)
Y = df["GPRC_RUS"]

### Fit the Model and Calculate Metrics
Define functions to train your model and use the scikit-learn modules to evaluate your results.

In [ ]:
def fit_model(X, Y):
    '''Use a RandomForestRegressor for this problem.'''
    
    # define the model to use
    model = RandomForestRegressor(criterion='squared_error', random_state=47)
    
    # Train the model
    model.fit(X, Y)
    
    return model

In [ ]:
def calculate_metrics(model, X_test_scaled, Y_test):
    '''Get model evaluation metrics on the test set.'''
    
    # Get model predictions
    y_predict_r = model.predict(X_test_scaled)
    
    # Calculate evaluation metrics for assesing performance of the model.
    r_square = r2_score(Y_test, y_predict_r)
    mse = mean_squared_error(Y_test, y_predict_r)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(Y_test, y_predict_r)
    
    return r_square, rmse, mae

In [ ]:
def train_and_get_metrics(X, Y):
    '''Train a Random Forest Regressor and get evaluation metrics'''
    
    # Split train and test sets
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 123)

    # Normalize all features of the train and test dataset here
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Call the fit model function to train the model on the normalized features and the target values
    model = fit_model(X_train_scaled, Y_train)

    # Make predictions on test dataset and calculate metrics
    r_square, rmse, mae = calculate_metrics(model, X_test_scaled, Y_test)

    return r_square, rmse, mae

In [ ]:
def evaluate_model_on_features(X, Y):
    '''Train model and display evaluation metrics.'''
    
    # Train the model, predict values and get metrics
    r_square, rmse, mae = train_and_get_metrics(X, Y)

    # Construct a dataframe to display metrics
    display_df = pd.DataFrame([[r_square, rmse, mae, X.shape[1]]], columns=["R-square", "RMSE", "MAE", 'Feature Count'])
    
    return display_df

Now train the model with all features included then calculate the metrics. This will be your baseline and you will compare this to the next outputs when you do feature selection.

In [ ]:
# Calculate evaluation metrics
all_features_eval_df = evaluate_model_on_features(X, Y)
all_features_eval_df.index = ['All features']

# Initialize results dataframe
results = all_features_eval_df

# Check the metrics
results.head()

## Correlation Matrix

It is a good idea to calculate and visualize the correlation matrix of a data frame to see which features have high correlation. The darker blue boxes show features with high positive correlation while white ones indicate high negative correlation.

In [ ]:
# Set figure size
plt.figure(figsize=(20,20))

# Calculate correlation matrix
cor = df.corr() 

# Plot the correlation matrix
sns.heatmap(cor, annot=True, cmap=plt.cm.PuBu)
plt.show()

### Correlation with the target variable

Let's start by determining which features are strongly correlated with GPRC_RUS (i.e. the target variable).

In [ ]:
# Get the absolute value of the correlation
cor_target = abs(cor["GPRC_RUS"])

# Select highly correlated features (thresold = 0.2)
relevant_features = cor_target[cor_target>0.2]

# Collect the names of the features
names = [index for index, value in relevant_features.items()]

# Drop the target variable from the results
names.remove('GPRC_RUS')

# Display the results
print(names)

Now try training the model again but only with the features in the columns you just gathered. Try to see if there is an improvement in the metrics compared to the earlier model.

In [ ]:
# Evaluate the model with new features
strong_features_eval_df = evaluate_model_on_features(df[names], Y)
strong_features_eval_df.index = ['Strong features']

# Append to results and display
# results = results.append(strong_features_eval_df)
results = pd.concat([results, strong_features_eval_df])
results.head()

### Correlation with other features

Now eliminate features which are highly correlated with each other. This helps remove redundant features thus resulting in a simpler model.
Plot the correlation matrix of the features selected previously.

In [ ]:
# Set figure size
plt.figure(figsize=(20,20))

# Calculate the correlation matrix for target relevant features that you previously determined
new_corr = df[names].corr()

# Visualize the correlation matrix
sns.heatmap(new_corr, annot=True, cmap=plt.cm.Blues)
plt.show()

Some features are indeed highly correlated. Now plot magnified view of the features that are highly correlated to each other.

In [ ]:
# Set figure size
plt.figure(figsize=(20,20))

# Select a subset of features
new_corr = df[['USDRUB', 'Fiscal revenues:Cumulative value', 'Gold reserves:Russia:Current month value', 'Fiscal Spending:Cumulative Value', 'Dollar Index', 'M2', 'CPI 2000=100', 'Combined Government Budgets:Revenues', 'Combined Government Budgets:Expenditures', 'PPI', 'UK Futures Official Price: LME 3 Months Nickel', 'UK futures closing price (electronic): LME 3-month nickel', 'UK Spot Settlement Price:LME Nickel', 'UK Spot Closing Price (OTC):LME Nickel', 'U.S. futures settlement price (active contract): CBOT corn', 'U.S. futures closing price (continuous):CBOT corn', 'U.S. futures closing price (active contract): CBOT corn', 'U.S. futures closing prices (continuous): NYMEX natural gas', 'U.S. Spot Price:Natural Gas:Henry Center Delivery', 'Official Futures Price: LME 3-month Aluminium', 'China Futures Closing Price (Continuous): Aluminum', 'China Futures Closing Price (Active Contract):Aluminum', 'China spot price:Aluminum:A00:National', 'UK London Spot Palladium: EUR/oz in EUR']].corr()

# Visualize the correlation matrix
sns.heatmap(new_corr, annot=True, cmap=plt.cm.Blues)
plt.show()

Now evaluate the model on the features selected based on your observations. You can see an improvement on the metrics with fewer features selected.

In [ ]:
# Remove the features with high correlation to other features
subset_feature_corr_names = [x for x in names if x not in ['Gold reserves:Russia:Current month value', 'Fiscal Spending:Cumulative Value', 'Dollar Index', 'M2', 'CPI 2000 = 100', 'Combined Government Budgets:Revenues', 'Combined Government Budgets:Expenditures', 'PPI', 'UK Futures Official Price: LME 3 Months Nickel','UK futures closing price (electronic): LME 3-month nickel','UK Spot Closing Price (OTC):LME Nickel','U.S. futures closing price (continuous):CBOT corn','U.S. futures closing price (active contract): CBOT corn', 'U.S. futures closing prices (continuous): NYMEX natural gas','China Futures Closing Price (Continuous): Aluminum','China Futures Closing Price (Active Contract):Aluminum','China spot price:Aluminum:A00:National','UK London Spot Palladium: EUR/oz in EUR']]

# Calculate and check evaluation metrics
subset_feature_eval_df = evaluate_model_on_features(df[subset_feature_corr_names], Y)
subset_feature_eval_df.index = ['Subset features']

# Append to results and display
# results = results.append(subset_feature_eval_df)
results = pd.concat([results, subset_feature_eval_df])
results.head(n=10)

### Univariate Selection with Sci-Kit Learn

Try a different filter method for feature selection using the ANOVA F-values to select the top 20 features using `SelectKBest()`.

In [ ]:
def univariate_selection():
    
    # Split train and test sets
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 123)
    
    # Normalize all features of the train and test dataset here
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # User SelectKBest to select top 20 features based on f-test
    selector = SelectKBest(f_classif, k=20)
    
    # Fit to scaled data, then transform it
    X_new = selector.fit_transform(X_train_scaled, Y_train)
    
    # Print the results
    feature_idx = selector.get_support()
    for name, included in zip(df.drop("GPRC_RUS", axis=1).columns, feature_idx):
        print("%s: %s" % (name, included))
    
    # Drop the target variable
    feature_names = df.drop("GPRC_RUS", axis=1).columns[feature_idx]
    
    return feature_names

You will now evaluate the model on the features selected by univariate selection.

In [ ]:
univariate_feature_names = univariate_selection()

In [ ]:
# Calculate and check model metrics
univariate_eval_df = evaluate_model_on_features(df[univariate_feature_names], Y)
univariate_eval_df.index = ['F-test']

# Append to results and display
# results = results.append(univariate_eval_df)
results = pd.concat([results, univariate_eval_df])
results.head(n=10)

In [ ]:
print(univariate_feature_names)

## Wrapper Methods

Wrapper methods use a model to measure the effectiveness of a particular subset of features. Try using **Recursive Feature Elimination** to prune the number of features based on feature importance scores. It wraps around the selected model **RandomForestRegressor** to perform feature selection. Repeat the same task of selecting the top 10 features using RFE instead of SelectKBest.

In [ ]:
def run_rfe():
    
    # Split train and test sets
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 123)
    
    # Normalize all features of the train and test dataset here.
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define the model
    model = RandomForestRegressor(criterion='squared_error', random_state=47)
    
    # Wrap RFE around the model
    rfe = RFE(model, n_features_to_select=20)
    
    # Fit RFE
    rfe = rfe.fit(X_train_scaled, Y_train)
    feature_names = df.drop("GPRC_RUS", axis=1).columns[rfe.get_support()]
    
    return feature_names

rfe_feature_names = run_rfe()

You will now evaluate the **RandomForestRegressor** on the features selected by RFE. Observe the new model performance.

In [ ]:
# Calculate and check model metrics
rfe_eval_df = evaluate_model_on_features(df[rfe_feature_names], Y)
rfe_eval_df.index = ['RFE']

# Append to results and display
# results = results.append(rfe_eval_df)
results = pd.concat([results, rfe_eval_df])
results.head(n=10)

In [ ]:
print(rfe_feature_names)

## Embedded Methods

Some models already have intrinsic properties that select the best features when it is constructed. With that, you can simply access these properties to get the scores for each feature.

### Feature Importances

**Feature importance** is already built-in in scikit-learn’s tree based models like **RandomForestRegressor**. Once the model is fit, the feature importance is available as a property named **feature_importances_**.

In [ ]:
def feature_importances_from_tree_based_model_():
    
    # Split train and test set
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 123)
    
    # Define the model to use
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = RandomForestRegressor()
    model = model.fit(X_train_scaled,Y_train)
    
    # Plot feature importance
    plt.figure(figsize=(10, 12))
    feat_importances = pd.Series(model.feature_importances_, index=X.columns)
    feat_importances.sort_values(ascending=False).plot(kind='barh')
    plt.show()
    
    return model


def select_features_from_model(model):
    
    model = SelectFromModel(model, prefit=True, threshold=0.02)
    feature_idx = model.get_support()
    feature_names = df.drop("GPRC_RUS", axis=1).columns[feature_idx]
        
    return feature_names

model = feature_importances_from_tree_based_model_()
feature_imp_feature_names = select_features_from_model(model)

In [ ]:
# Calculate and check model metrics
feat_imp_eval_df = evaluate_model_on_features(df[feature_imp_feature_names], Y)
feat_imp_eval_df.index = ['Feature Importance']

# Append to results and display
# results = results.append(feat_imp_eval_df)
results = pd.concat([results, feat_imp_eval_df])
results.head(n=10)

In [ ]:
print(feature_imp_feature_names)

With these results and also some domain knowledge, you can decide which set of features to use to train on the entire dataset. On the other hand, if you find that all the resulting scores for all approaches are acceptable, then you may just go for the method with the smallest set of features.

## Find Feature Importances of different regression models

In [ ]:
# Train / Test split
# Split train and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 123)

# Normalize all features of the train and test dataset here
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# define the model to use
model = RandomForestRegressor(criterion='squared_error', random_state=47)

# Train the model
model.fit(X_train_scaled, Y_train)

# Print the mean accuracy achieved by the regressor on the test set
model.score(X_test_scaled, Y_test)

In [ ]:
#collect feature_names
feature_names = [index for index, value in X.items()]

feature_names

In [ ]:
from sklearn.inspection import permutation_importance

def feature_importance(model, X, Y, top_limit=None):

  # Retrieve the Bunch object after 50 repeats
  # n_repeats is the number of times that each feature was permuted to compute the final score
  bunch = permutation_importance(model, X, Y,
                                 n_repeats=50, random_state=42)

  # Average feature importance
  imp_means = bunch.importances_mean

  # List that contains the index of each feature in descending order of importance
  ordered_imp_means_args = np.argsort(imp_means)[::-1]

  # If no limit print all features
  if top_limit is None:
    top_limit = len(ordered_imp_means_args)

  # Print relevant information
  for i, _ in zip(ordered_imp_means_args, range(top_limit)):
    name = feature_names[i]
    imp_score = imp_means[i]
    imp_std = bunch.importances_std[i]
    print(f"Feature {name} with index {i} has an average importance score of {imp_score:.3f} +/- {imp_std:.3f}\n")

The importance score is computed in a way that higher values represent better predictive power.

Now use the `feature_importance` function on the Random Forest Regressor and the train set & test set:

In [ ]:
feature_importance(model, X_train_scaled, Y_train)

In [ ]:
feature_importance(model, X_test_scaled, Y_test)

In [ ]:
print("On TRAIN split:\n")
feature_importance(model, X_train_scaled, Y_train, top_limit=10)

print("\nOn TEST split:\n")
feature_importance(model, X_test_scaled, Y_test, top_limit=10)

In [ ]:
# Preserve the top 6 features
X_train_top_features = X_train_scaled[:,[1, 2, 8, 25, 34, 37]]
X_test_top_features = X_test_scaled[:,[1, 2, 8, 25, 34, 37]]

# Re-train with only these features
model_top = RandomForestRegressor(criterion='squared_error', random_state=47).fit(X_train_top_features, Y_train)

# Compute mean accuracy achieved
model_top.score(X_test_top_features, Y_test)

Try out other regression methods

In [ ]:
from sklearn.svm import SVR
from sklearn.linear_model import Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor

# Select 4 new classifiers
clfs = {"Lasso": Lasso(alpha=0.05), 
        "Ridge": Ridge(), 
        "Decision Tree": DecisionTreeRegressor(), 
        "Support Vector": SVR()}


# Compute feature importance on the test set given a classifier
def fit_compute_importance(clf):
  clf.fit(X_train_scaled, Y_train)
  print(f"📏 Mean accuracy score on the test set: {clf.score(X_test_scaled, Y_test)*100:.2f}%\n")
  print("🔝 Top 6 features when using the test set:\n")
  feature_importance(clf, X_test_scaled, Y_test, top_limit=6)


# Print results
for name, clf in clfs.items():
  print("====="*20)
  print(f"➡️ {name} regressor\n")
  fit_compute_importance(clf)